## Reorder data

In [1]:
import argparse
import time
import gc
import os
import glob
import json
import re
from os.path import join as pjoin

raw_path = "../bert_data/id-p2"
result_path = "../results/baseline_4"
language = "indonesia"
long = False
use_valid_data = True

for f in glob.glob(pjoin(result_path, 'ro.*.*')):
    if os.path.isfile(f):
        print(f"Removing {f}...")
        os.remove(f)

corpus_type = "valid" if use_valid_data else "test"
print("Corpus type:", corpus_type)

# Map seluruh dokumen test set ke index 
mapping_tgt = {}
threshold = 0    
for json_f in sorted(glob.glob(pjoin(raw_path, f'*{corpus_type}.*.json'))):
    with open(json_f) as json_f:
        data = json.load(json_f)
        threshold = threshold + len(data)
        for i in range(len(data)):
            j = i + threshold - len(data)
            if len(mapping_tgt) < threshold: 
                tgt = data[i]['tgt_txt']
                mapping_tgt[j] = tgt
          
print("Total data:", len(mapping_tgt))
keys = list(mapping_tgt.keys())
vals = list(mapping_tgt.values())

gold_docs = []
cand_docs = []
src_docs = []

for f in glob.glob(pjoin(result_path, '*.*.gold')):
    real_name = f.split("/")[-1].split(".")[0]
    step = f.split("/")[-1].split(".")[1]
    with open(f) as file:
        for line in file: 
            line = line.strip()
            gold_docs.append(line) 
            
for f in glob.glob(pjoin(result_path, '*.*.candidate')):
    with open(f) as file:
        for line in file: 
            line = line.strip() 
            cand_docs.append(line) 
            
for f in glob.glob(pjoin(result_path, '*.*.raw_src')):
    with open(f) as file:
        for line in file: 
            line = line.strip() 
            src_docs.append(line) 

save_gold = ["" for i in range(len(gold_docs))]
save_cand = ["" for i in range(len(gold_docs))]
save_src = ["" for i in range(len(gold_docs))]

print("Matching the data...")

for i in range(len(gold_docs)):    # looping seluruh gold summary
    if gold_docs[i] in vals:       # jika gold doc nya ada di dalam test set
        pos = vals.index(gold_docs[i])
        save_gold[pos] = gold_docs[i]  # tempatkan gold sesuai key di dalam dict test set
        save_cand[pos] = cand_docs[i]
        save_src[pos] = src_docs[i]

print("Saving the reordered data...")

with open(result_path + "/ro." + real_name + "." + step + ".gold", 'w') as f:
    for e in save_gold:
        f.write(f"{e}\n")

with open(result_path + "/ro." + real_name + "." + step +".candidate", 'w') as f:
        for e in save_cand:
            f.write(f"{e}\n")

with open(result_path + "/ro." + real_name + "." + step + ".raw_src", 'w') as f:
        for e in save_src:
            f.write(f"{e}\n")

Removing ../results/baseline_4/ro.result.9000.gold...
Removing ../results/baseline_4/ro.result.9000.truncate...
Removing ../results/baseline_4/ro.result.9000.rouge...
Removing ../results/baseline_4/ro.result.9000.raw_src...
Removing ../results/baseline_4/ro.result.9000.candidate...
Corpus type: valid
Total data: 4780
Matching the data...
Saving the reordered data...


## Get ROUGE scores per docs

In [2]:
import rouge 

In [3]:
""" Produce separated file containing ROUGE results for each doc in test set. """

file_path = "../results/baseline_4/ro.result.9000"

aggregator = "Individual"
print('Evaluation with {}'.format(aggregator))

candidates = []
with open(file_path + ".candidate") as file:
    for line in file:
        line = line.strip()
        candidates.append(line)
        
golds = []
with open(file_path + ".gold") as file:
    for line in file:
        line = line.strip()
        golds.append(line)

# logger.info(f"Candidates length: {len(candidates)}")
# logger.info(f"Golds length: {len(golds)}")

evaluator = rouge.Rouge(metrics=['rouge-n', 'rouge-l'],
                       max_n=3,
                       limit_length=False,
                       alpha=0.5, # Default F1_score
                       apply_avg=False,
                       apply_best=False,
                       weight_factor=1.0,
                       stemming=True)

scores = evaluator.get_scores(candidates, golds)

results_list = [0 for i in range(len(candidates))]
for metric, results in sorted(scores.items(), key=lambda x: x[0]):
    for cand_id, results_per_ref in enumerate(results):
        if metric == 'rouge-1':
            results_list[cand_id] = [results_per_ref['f'][0]]
        else:
            results_list[cand_id].append(results_per_ref['f'][0])

with open(file_path + ".rouge", "w") as file:
    for res in results_list:
        line = 'ROUGE-F(1/2/3/L): {:2.4f}/{:2.4f}/{:2.4f}/{:2.4f}'.format(res[0]*100,res[1]*100,res[2]*100,res[3]*100)
        file.write(f"{line}\n")
        # logger.info(line)

Evaluation with Individual


## Truncate summary

In [4]:
result_path = "../results/baseline_4"
files = []

for file in os.listdir(result_path):
    if "ro." in file:
        files.append(file)

lengths_gold = []
lengths_cand = []
for file in files:
    if "gold" in file:
        with open(f"{result_path}/{file}") as file:
            for line in file:
                lengths_gold.append(len(line.split()))   
    elif "candidate" in file:
        with open(f"{result_path}/{file}") as file:
            for line in file:
                lengths_cand.append(len(line.split())) 

avg_words_gold = sum(lengths_gold) / len(lengths_gold)
avg_words_cand = sum(lengths_cand) / len(lengths_cand)

print(f"{language.upper()}")
print("Average words in gold:", avg_words_gold)
print("Average words in candidate:", avg_words_cand)
print("Max. words in gold:", max(lengths_gold))
print("Max. words in candidate:", max(lengths_cand))

INDONESIA
Average words in gold: 23.939539748953976
Average words in candidate: 39.1255230125523
Max. words in gold: 46
Max. words in candidate: 47


In [5]:
if avg_words_cand > avg_words_gold + 5:
    max_words = round(avg_words_gold) + 5
    truncates = []
    for file in files:
        if "candidate" in file:
            with open(f"{result_path}/{file}") as file:
                for line in file:
                    truncates.append(" ".join(line.split()[:max_words]))
                    
    truncate_filename = ".".join(file.name.split(".")[:-1]) + ".truncate"
    
    with open(f"{truncate_filename}", "w") as file:
        for trunc in truncates:
            file.write(f"{trunc}\n")

## Handle Repetition

In [6]:
import string

for file in files:
    if "candidate" in file:
        lines_handled = []
        
        with open(f"{result_path}/{file}") as f:
            for line in f:
                
                text = line.split()
                
                for i in range(len(text)):
                    if i > 0:
                        if text[i] == text[i - 1] and text[i] in string.punctuation:
                            text[i-1] = ""
                        if text[i] in string.punctuation and text[i-1] in string.punctuation:
                            text[i] = ""
                        if text[i] == text[i - 1]:
                            text[i-1] = ""
                # print(text)
                text = ' '.join(text)
                # print(' '.join(text.split()))
                text = ' '.join(text.split())
                lines_handled.append(text)
                
        with open(f"{result_path}/{file}", "w") as f:     
            for line in lines_handled:
                f.write(f"{line}\n")